# 信用卡欺诈检测 — 金额感知损失函数与迁移实验

本 notebook 独立验证金额感知损失函数，不修改主 notebook 的既有结果。

## 阅读方式

- **§0–§4**：公式、数据协议、复用函数、权重实现与业务指标。
- **§5**：第一阶段，在 BASE 特征上依次执行实验 0、1、2、3。
- **§6**：从阶段一结果中输出 Pareto 前沿并选择暂定参数。
- **§7**：第二阶段，执行“BASE / §7–8 胜者 × 标准 / 新损失”的 2×2 迁移实验。
- 所有输出写入 `src/output/amount_aware_loss/`，不覆盖 `final_report/`。

完整跑批计算量较大，默认关闭。先运行所有 prep cell，再将顶部的
`RUN_STAGE1` 或 `RUN_STAGE2` 改为 `True`。每组 OOF 会单独缓存，中断后可继续。


In [130]:
# §0 导入依赖
%matplotlib inline
from pathlib import Path
import hashlib
import json
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve
import lightgbm as lgb
import xgboost as xgb
import optuna
from purgedcv import WalkForwardSplit
from purgedcv.diagnostics import assert_no_temporal_leakage
from IPython.display import display

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 150)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Heiti TC', 'PingFang SC', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


## 0. V2 研究设计：先修比较协议，再扩大搜索

V1 的 `matchedFP` 只在 calibration 匹配 FP，迁移到 evaluation 后曾出现 7 个预算对应 47,664 个 FP，不能用于选参。V2 改用固定告警量 **Top-K**：所有候选在同一数据段只比较风险分数最高的 K 笔，因此工作量严格相同，不依赖概率刻度能否跨时间迁移。

参数搜索只使用 OOF calibration 段的 `AmountRecall@K`、`MicroRecall@K` 与 AUC-PR；evaluation 段只用于最终候选验收。

训练协议也统一：neutral 和金额感知候选都只给 fit 子集传 `sample_weight`，早停集统一使用未加权 logloss。实验 0 仍保留原 notebook 的内置 balanced 模型作为锚点，并检查 neutral 是否足够接近。

执行顺序：粗网格实验 0–3 → 局部精细网格 → Optuna 多目标搜索 → 合并 calibration Pareto 候选 → 第二阶段 2×2 迁移。


In [131]:
# §0 V2 参数与路径
RUN_COARSE_GRID = True
RUN_FINE_GRID = True
RUN_OPTUNA = True
RUN_STAGE2 = True
# RUN_STABILITY = os.environ.get('RUN_STABILITY', '0') == '1'
RUN_STABILITY = 1
FORCE_RERUN = True
OPTUNA_N_TRIALS = 30
PARITY_AUC_TOL = 0.01
STABILITY_SEEDS = [42, 52, 62, 72, 82]
BOOTSTRAP_N = 2000
BOOTSTRAP_RANDOM_STATE = 20260707

MODELS = ['LightGBM', 'XGBoost']
BETA_GRID = [0.25, 0.5, 1.0, 2.0, 4.0]
ETA_GRID = [0.25, 0.5, 1.0, 2.0]
KAPPA_GRID = [0.5, 0.75, 1.0, 1.25, 1.5]

CV_N_SPLITS = 5
CV_RANDOM_STATE = 42
CV_EMBARGO = pd.Timedelta(hours=2)
CV_PURGE_HORIZON = pd.Timedelta(0)
EARLY_STOPPING_ROUNDS = 50
MAX_BOOST_ROUNDS = 1500
ES_FRAC = 0.20
ES_MIN_FRAUD = 5
ES_MAX_FRAC = 0.35
OOF_CAL_FRAC = 0.25
OOF_CAL_MIN_FRAUD = 50
OOF_CAL_MAX_FRAC = 0.40

ABLATION_WINNER_LABEL = 'IF+Ed[log1p+one_euro+micro+bands]+A_top2(V14+V4)'


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'input' / 'creditcard.csv').exists():
            return candidate
    raise FileNotFoundError('无法找到 input/creditcard.csv，请从项目内启动 notebook')


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
DATA_PATH = PROJECT_ROOT / 'input' / 'creditcard.csv'
FINAL_REPORT_DIR = SRC_DIR / 'output' / 'final_report'
OUT_AMOUNT_LOSS = SRC_DIR / 'output' / 'amount_aware_loss_v2'
OUT_COARSE = OUT_AMOUNT_LOSS / '01_coarse_grid'
OUT_FINE = OUT_AMOUNT_LOSS / '02_fine_grid'
OUT_OPTUNA = OUT_AMOUNT_LOSS / '03_optuna'
OUT_STAGE2 = OUT_AMOUNT_LOSS / '04_stage2_transfer'
OUT_STABILITY = OUT_AMOUNT_LOSS / '05_stability'
STABILITY_PRED_DIR = OUT_STABILITY / 'predictions'
OOF_CACHE_DIR = OUT_AMOUNT_LOSS / 'oof_cache'
for _d in (OUT_AMOUNT_LOSS, OUT_COARSE, OUT_FINE, OUT_OPTUNA, OUT_STAGE2,
           OUT_STABILITY, STABILITY_PRED_DIR, OOF_CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('V2 独立输出目录:', OUT_AMOUNT_LOSS)


PROJECT_ROOT: /Users/jingyuhe/Desktop/credit-fraud-dealing-with-imbalanced-datasets
V2 独立输出目录: /Users/jingyuhe/Desktop/credit-fraud-dealing-with-imbalanced-datasets/src/output/amount_aware_loss_v2


## 1. 统一损失函数：基础欺诈价值 + 金额价值 + 卡测试保护

对每个训练折的 **fit 子集**，定义类别不平衡比：

$$r=\frac{N_0}{N_1}$$

金额函数使用对数、欺诈样本 95% 分位截尾，并缩放到 $[0,1]$：

$$g(A_i)=\frac{\min[\log(1+A_i),q_{95}^{+}]}{q_{95}^{+}}$$

小额卡测试指示变量：

$$h(A_i)=\mathbb I(A_i\le1)$$

欺诈内部业务权重：

$$u_i=1+\beta g(A_i)+\eta h(A_i)$$

其中基础项 1 保证零金额和小额欺诈不会被忽略；$\beta$ 只控制大额欺诈倾向；$\eta$ 只控制小额卡测试保护。为了避免改变 $\beta,\eta$ 时连带改变整个正类总权重，使用 fit 子集正类均值归一化：

$$\bar u_+=\frac1{N_1}\sum_{j:y_j=1}u_j$$

最终样本权重为：

$$w_i=\begin{cases}1,&y_i=0\\
\kappa r\dfrac{u_i}{\bar u_+},&y_i=1\end{cases}$$

损失函数是加权二元交叉熵：

$$\boxed{\mathcal L=-\frac1n\sum_iw_i[y_i\log p_i+(1-y_i)\log(1-p_i)]}$$

- $r$：处理类别不平衡，对应原模型的 balanced 权重。
- $\kappa$：调整整个欺诈类相对于正常类的重要性。
- $\beta$：在欺诈类内部向大额交易重新分配权重。
- $\eta$：在欺诈类内部向 `Amount<=1` 的卡测试交易重新分配权重。
- $\bar u_+$：让 $\beta,\eta$ 只改变正类内部结构，不偷偷放大正类总权重。

**防泄漏约束：** $r,q_{95}^{+},\bar u_+$ 只能在每折 fit 子集计算；早停集和验证折只能使用已有状态 transform。

失败原因：虽然损失函数用了银行的惯例做法，但银行不会只靠 Amount 区分欺诈。由于所有特征业务含义已被 PCA 脱敏破坏，哪怕照搬银行的方法也得不到好的效果。


In [132]:
# §1 prep：金额感知权重
def fit_weight_state(amount, y, beta, eta, kappa, clip_quantile=0.95, micro_max=1.0):
    amount = np.asarray(amount, dtype=float)
    y = np.asarray(y, dtype=int)
    if len(amount) != len(y):
        raise ValueError('amount 与 y 长度不一致')
    if beta < 0 or eta < 0 or kappa <= 0:
        raise ValueError('要求 beta>=0、eta>=0、kappa>0')
    pos = y == 1
    n_pos, n_neg = int(pos.sum()), int((~pos).sum())
    if n_pos == 0 or n_neg == 0:
        raise ValueError('fit 子集必须同时包含正常与欺诈样本')

    log_amount = np.log1p(np.clip(amount, 0.0, None))
    q95_pos = float(np.quantile(log_amount[pos], clip_quantile))
    q95_pos = max(q95_pos, 1e-12)
    g = np.clip(log_amount / q95_pos, 0.0, 1.0)
    h = (amount <= micro_max).astype(float)
    u = 1.0 + float(beta) * g + float(eta) * h
    u_bar_pos = float(u[pos].mean())
    return {
        'beta': float(beta), 'eta': float(eta), 'kappa': float(kappa),
        'clip_quantile': float(clip_quantile), 'micro_max': float(micro_max),
        'q95_pos': q95_pos, 'u_bar_pos': u_bar_pos,
        'class_ratio': float(n_neg / n_pos),
        'n_pos_fit': n_pos, 'n_neg_fit': n_neg,
    }


def transform_business_weight(amount, y, state):
    amount = np.asarray(amount, dtype=float)
    y = np.asarray(y, dtype=int)
    log_amount = np.log1p(np.clip(amount, 0.0, None))
    g = np.clip(log_amount / state['q95_pos'], 0.0, 1.0)
    h = (amount <= state['micro_max']).astype(float)
    u = 1.0 + state['beta'] * g + state['eta'] * h
    weight = np.ones(len(y), dtype=float)
    pos = y == 1
    weight[pos] = (
        state['kappa'] * state['class_ratio']
        * u[pos] / state['u_bar_pos']
    )
    if not np.isfinite(weight).all() or (weight <= 0).any():
        raise ValueError('出现非有限或非正样本权重')
    return weight


In [133]:
# §1 exec：权重不变量自检（不训练模型）
_amount = np.array([0.0, 1.0, 10.0, 1000.0, 20.0, 30.0])
_y = np.array([1, 1, 1, 1, 0, 0])

_neutral = fit_weight_state(_amount, _y, beta=0, eta=0, kappa=1)
_w0 = transform_business_weight(_amount, _y, _neutral)
assert np.allclose(_w0[_y == 0], 1.0)
assert np.allclose(_w0[_y == 1], _neutral['class_ratio'])

_amount_state = fit_weight_state(_amount, _y, beta=2, eta=0, kappa=1)
_wa = transform_business_weight(_amount, _y, _amount_state)
assert _wa[3] > _wa[2]  # 大额欺诈比中额欺诈权重高

_micro_state = fit_weight_state(_amount, _y, beta=0, eta=2, kappa=1)
_wm = transform_business_weight(_amount, _y, _micro_state)
assert _wm[0] > _wm[2] and _wm[1] > _wm[2]
print('权重公式自检通过')
display(pd.DataFrame({'Amount': _amount, 'Class': _y, 'neutral': _w0, 'amount_weighted': _wa, 'micro_weighted': _wm}))


权重公式自检通过


,Amount,Class,neutral,amount_weighted,micro_weighted
0,0.0,1,0.5,0.286042,0.75
1,1.0,1,0.5,0.349671,0.75
2,10.0,1,0.5,0.506160,0.25
3,1000.0,1,0.5,0.858127,0.25
4,20.0,0,1.0,1.000000,1.00
5,30.0,0,1.0,1.000000,1.00


## 2. 数据与 §7/8 胜者解析

为保证 2×2 实验完全对齐，BASE 与胜者都从主 notebook 已保存的 `df_features.pkl` 读取。这里只读取主报告结果，不写回 `final_report/`。

注意：`runtime_state.json` 中的 `WINNER_FEATURES` 可能已被后续 FE8 实验更新，因此本 notebook 不直接使用它；而是按 §7/8 固定标签从 `ablation_combo_specs.csv` 精确解析 8 个额外列。


In [134]:
# §2 prep：数据与特征组
def resolve_ablation_extra(label):
    path = FINAL_REPORT_DIR / '07_ablation' / 'ablation_combo_specs.csv'
    if not path.is_file():
        raise FileNotFoundError(f'缺少 §7 输出: {path}')
    specs = pd.read_csv(path)
    row = specs[specs['label'].astype(str).str.strip() == label]
    if row.empty:
        raise KeyError(f'未找到 §7/8 胜者: {label}')
    return [c.strip() for c in str(row.iloc[0]['extra_cols']).split('|') if c.strip()]


state = json.loads((FINAL_REPORT_DIR / 'runtime_state.json').read_text(encoding='utf-8'))
BASE_FEATURES = list(state['BASE_FEATURES'])
WINNER_EXTRA = resolve_ablation_extra(ABLATION_WINNER_LABEL)
WINNER_FEATURES = BASE_FEATURES + WINNER_EXTRA

feature_path = FINAL_REPORT_DIR / 'df_features.pkl'
if not feature_path.is_file():
    raise FileNotFoundError(f'请先运行主 notebook 到 §7: {feature_path}')
df_all = pd.read_pickle(feature_path).sort_values('Time', kind='mergesort').reset_index(drop=True)
missing = [c for c in WINNER_FEATURES + ['Class', 'Amount', 'Time'] if c not in df_all.columns]
if missing:
    raise KeyError(f'df_features 缺列: {missing}')

print('样本数:', len(df_all), '| 欺诈数:', int(df_all['Class'].sum()))
print('BASE:', len(BASE_FEATURES), '列')
print('§7/8 胜者:', ABLATION_WINNER_LABEL)
print('额外列:', WINNER_EXTRA)


样本数: 284807 | 欺诈数: 492
BASE: 30 列
§7/8 胜者: IF+Ed[log1p+one_euro+micro+bands]+A_top2(V14+V4)
额外列: ['if_oof_score', 'log1p_amount', 'is_one_euro', 'is_micro_testing', 'is_amount_1_30', 'is_amount_75_110', 'one_euro_V14', 'one_euro_V4']


## 3. 统一训练与早停协议

- CV、embargo 与时间早停切分沿用主 notebook。
- `builtin` 只用于复现原始 baseline。
- `neutral` 与所有金额感知候选统一关闭模型内置类别权重，类别不平衡和业务权重全部由 fit 子集的 `sample_weight` 表达。
- 早停集不再传样本权重，所有方案统一用未加权 logloss 选择迭代轮数，避免“改变损失”同时改变早停评价口径。


In [135]:
# §3 prep：时间 CV
def build_cv_timestamps(data):
    t = pd.to_timedelta(data['Time'].astype(float), unit='s')
    return t.copy(), t.copy()


def iter_purged_cv_folds(data, n_splits=CV_N_SPLITS):
    n = len(data)
    pred, evalu = build_cv_timestamps(data)
    test_size = max(1, n // (n_splits + 1))
    cv = WalkForwardSplit(
        n_splits=n_splits, test_size=test_size, window='expanding',
        prediction_times=pred, evaluation_times=evalu,
        purge_horizon=CV_PURGE_HORIZON, embargo=CV_EMBARGO,
    )
    for tr_idx, va_idx in cv.split(np.arange(n)):
        assert_no_temporal_leakage(
            tr_idx, va_idx, prediction_times=pred, evaluation_times=evalu,
            purge_horizon=CV_PURGE_HORIZON,
        )
        yield np.asarray(tr_idx), np.asarray(va_idx)


def temporal_es_masks(y, es_frac=ES_FRAC, min_fraud_es=ES_MIN_FRAUD, max_frac=ES_MAX_FRAC):
    y = np.asarray(y, dtype=int)
    fraud_idx = np.flatnonzero(y == 1)
    norm_idx = np.flatnonzero(y == 0)
    if len(fraud_idx) < 2:
        n_es = max(1, min(len(y) - 1, int(len(y) * es_frac)))
        es = np.zeros(len(y), dtype=bool)
        es[-n_es:] = True
        return ~es, es
    frac = es_frac
    while frac <= max_frac + 1e-9:
        n_fraud_es = min(max(min_fraud_es, int(len(fraud_idx) * frac)), len(fraud_idx) - 1)
        n_norm_es = min(max(1, int(len(norm_idx) * frac)), len(norm_idx) - 1)
        es = np.zeros(len(y), dtype=bool)
        es[fraud_idx[-n_fraud_es:]] = True
        es[norm_idx[-n_norm_es:]] = True
        if int(y[es].sum()) >= min(min_fraud_es, len(fraud_idx) - 1):
            return ~es, es
        frac += 0.05
    raise ValueError('无法构造含足够欺诈样本的时间早停集')


In [136]:
# §3 prep：模型工厂与统一早停
def make_classifier(model_name, y_train, random_state, weight_scheme='balanced'):
    y_train = np.asarray(y_train, dtype=int)
    spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    if model_name == 'LightGBM':
        params = dict(
            n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6,
            num_leaves=31, min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=random_state, verbose=-1, n_jobs=-1,
        )
        if weight_scheme == 'balanced':
            params['class_weight'] = 'balanced'
        elif weight_scheme != 'no_weight':
            raise ValueError(weight_scheme)
        return lgb.LGBMClassifier(**params)
    if weight_scheme not in ('balanced', 'no_weight'):
        raise ValueError(weight_scheme)
    return xgb.XGBClassifier(
        n_estimators=MAX_BOOST_ROUNDS, learning_rate=0.05, max_depth=6,
        min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        random_state=random_state, eval_metric='logloss', verbosity=0, n_jobs=-1,
        scale_pos_weight=spw if weight_scheme == 'balanced' else 1.0,
    )


def fit_classifier(clf, model_name, X_fit, y_fit, X_es, y_es,
                   sample_weight=None, eval_sample_weight=None):
    # eval_sample_weight 参数仅保留 API 兼容；V2 明确不用于早停。
    kwargs = {'eval_set': [(X_es, y_es)]}
    if sample_weight is not None:
        kwargs['sample_weight'] = sample_weight
    if model_name == 'LightGBM':
        kwargs['eval_metric'] = 'binary_logloss'
        kwargs['callbacks'] = [lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)]
    else:
        kwargs['verbose'] = False
    clf.fit(X_fit, y_fit, **kwargs)
    return clf


In [137]:
# §3 prep：标准、neutral 与金额感知 OOF
def cross_val_builtin_oof(model_name, data, feature_cols, random_state=CV_RANDOM_STATE):
    X, y = data[feature_cols], data['Class'].astype(int)
    oof = np.full(len(data), np.nan, dtype=float)
    fold_ap = []
    for fold, (tr, va) in enumerate(iter_purged_cv_folds(data), start=1):
        fit_mask, es_mask = temporal_es_masks(y.iloc[tr])
        fit_idx, es_idx = tr[fit_mask], tr[es_mask]
        clf = make_classifier(model_name, y.iloc[fit_idx], random_state + fold, weight_scheme='balanced')
        fit_classifier(clf, model_name, X.iloc[fit_idx], y.iloc[fit_idx], X.iloc[es_idx], y.iloc[es_idx])
        p = clf.predict_proba(X.iloc[va])[:, 1]
        oof[va] = p
        fold_ap.append(float(average_precision_score(y.iloc[va], p)))
        print(f'  {model_name} builtin fold {fold}/{CV_N_SPLITS} 完成')
    return oof, fold_ap


def cross_val_weighted_oof(model_name, data, feature_cols, beta, eta, kappa,
                           random_state=CV_RANDOM_STATE):
    X, y = data[feature_cols], data['Class'].astype(int)
    amount = data['Amount'].astype(float)
    oof = np.full(len(data), np.nan, dtype=float)
    fold_ap, weight_states = [], []
    for fold, (tr, va) in enumerate(iter_purged_cv_folds(data), start=1):
        fit_mask, es_mask = temporal_es_masks(y.iloc[tr])
        fit_idx, es_idx = tr[fit_mask], tr[es_mask]
        state = fit_weight_state(amount.iloc[fit_idx], y.iloc[fit_idx], beta, eta, kappa,
                                 clip_quantile=0.95)
        sw_fit = transform_business_weight(amount.iloc[fit_idx], y.iloc[fit_idx], state)
        clf = make_classifier(model_name, y.iloc[fit_idx], random_state + fold, weight_scheme='no_weight')
        fit_classifier(
            clf, model_name, X.iloc[fit_idx], y.iloc[fit_idx], X.iloc[es_idx], y.iloc[es_idx],
            sample_weight=sw_fit,
        )
        p = clf.predict_proba(X.iloc[va])[:, 1]
        oof[va] = p
        fold_ap.append(float(average_precision_score(y.iloc[va], p)))
        weight_states.append(state)
        print(f'  {model_name} weighted fold {fold}/{CV_N_SPLITS} 完成 '
              f'(beta={beta:.4g}, eta={eta:.4g}, kappa={kappa:.4g})')
    return oof, fold_ap, weight_states


## 4. Top-K 固定告警量指标与窄表

`Top-K` 表示只审核风险分数最高的 K 笔交易。baseline 在 calibration 上用最佳 F1 阈值得到告警率 $K_{cal}/N_{cal}$；候选在 calibration 使用相同 $K_{cal}$，evaluation 使用相同比例对应的 $K_{eval}$。

这样每个模型的告警工作量完全一致，不会再出现 `matchedFP` 从 calibration 迁移到 evaluation 后失控的问题。参数只按 `*_cal_atK` 选；`*_eval_atK` 只用于最终验收。

notebook 不再展示 TP、TN、FraudAmount、MicroFN 等冗余列；它们仍保存在完整 CSV 供审计。


In [138]:
# §4 prep：cal/eval、F1 与 Top-K
def oof_cal_mask(y, cal_frac=OOF_CAL_FRAC, min_fraud_cal=OOF_CAL_MIN_FRAUD,
                 max_frac=OOF_CAL_MAX_FRAC):
    y = np.asarray(y, dtype=int)
    fraud_idx, norm_idx = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    frac = cal_frac
    while frac <= max_frac + 1e-9:
        nf = min(max(min_fraud_cal, int(len(fraud_idx) * frac)), len(fraud_idx) - 1)
        nn = min(max(1, int(len(norm_idx) * frac)), len(norm_idx) - 1)
        mask = np.zeros(len(y), dtype=bool)
        mask[fraud_idx[:nf]] = True
        mask[norm_idx[:nn]] = True
        if int(y[mask].sum()) >= min(min_fraud_cal, len(fraud_idx) - 1):
            return mask
        frac += 0.05
    raise ValueError('无法构造 OOF calibration 段')


def best_f1_threshold(y, p):
    precision, recall, threshold = precision_recall_curve(y, p)
    if not len(threshold):
        return 0.5
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(threshold[int(np.nanargmax(f1))])


def confusion(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return dict(TN=int(tn), FP=int(fp), FN=int(fn), TP=int(tp),
                Precision=float(precision), Recall=float(recall), F1=float(f1))


def metrics_at_top_k(y, amount, p, k, suffix):
    y = np.asarray(y, dtype=int)
    amount = np.asarray(amount, dtype=float)
    p = np.asarray(p, dtype=float)
    k = int(np.clip(int(k), 1, len(y)))
    order = np.argsort(-p, kind='mergesort')
    pred = np.zeros(len(y), dtype=bool)
    pred[order[:k]] = True
    out = confusion(y, pred)
    fraud, caught = y == 1, (y == 1) & pred
    total_amount = float(amount[fraud].sum())
    caught_amount = float(amount[caught].sum())
    micro = fraud & (amount <= 1.0)
    micro_tp = int((micro & pred).sum())
    out.update({
        'K': k,
        'AmountRecall': caught_amount / total_amount if total_amount else np.nan,
        'MissedAmount': total_amount - caught_amount,
        'MicroRecall': micro_tp / int(micro.sum()) if micro.any() else np.nan,
        'MicroFN': int(micro.sum()) - micro_tp,
        'FraudAmount': total_amount,
    })
    assert out['TP'] + out['FP'] == k
    return {f'{name}_{suffix}_atK': value for name, value in out.items()}


In [139]:
# §4 prep：完整评价与窄表展示
def evaluate_oof_business(data, oof, fold_ap, alert_rate=None):
    finite = np.isfinite(oof)
    y = data.loc[finite, 'Class'].astype(int).to_numpy()
    amount = data.loc[finite, 'Amount'].astype(float).to_numpy()
    p = np.asarray(oof, dtype=float)[finite]
    cal, ev = oof_cal_mask(y), None
    ev = ~cal
    t_f1 = best_f1_threshold(y[cal], p[cal])
    if alert_rate is None:
        alert_rate = float((p[cal] >= t_f1).mean())
    k_cal = max(1, int(round(alert_rate * int(cal.sum()))))
    k_eval = max(1, int(round(alert_rate * int(ev.sum()))))
    result = {
        'AUC-PR_mean': float(np.mean(fold_ap)), 'AUC-PR_std': float(np.std(fold_ap)),
        'AUC-PR_cal': float(average_precision_score(y[cal], p[cal])),
        'AUC-PR_eval': float(average_precision_score(y[ev], p[ev])),
        'alert_rate': float(alert_rate), 'n_cal': int(cal.sum()), 'n_eval': int(ev.sum()),
        'n_fraud_cal': int(y[cal].sum()), 'n_fraud_eval': int(y[ev].sum()),
    }
    result.update(metrics_at_top_k(y[cal], amount[cal], p[cal], k_cal, 'cal'))
    result.update(metrics_at_top_k(y[ev], amount[ev], p[ev], k_eval, 'eval'))
    return result


def compact_classification_table(frame, split='eval'):
    cols = ['feature_group', 'model', 'experiment', 'loss_kind', 'beta', 'eta', 'kappa',
            f'AUC-PR_{split}', f'F1_{split}_atK', f'Precision_{split}_atK',
            f'Recall_{split}_atK', f'FP_{split}_atK', f'FN_{split}_atK']
    rename = {'feature_group': '特征组', 'model': '模型', 'experiment': '实验',
              'loss_kind': '损失', 'beta': 'β', 'eta': 'η', 'kappa': 'κ',
              f'AUC-PR_{split}': 'AUC-PR', f'F1_{split}_atK': 'F1@K',
              f'Precision_{split}_atK': 'Precision@K', f'Recall_{split}_atK': 'Recall@K',
              f'FP_{split}_atK': 'FP@K', f'FN_{split}_atK': 'FN@K'}
    out = frame[[c for c in cols if c in frame.columns]].copy()
    out['experiment'] = out['experiment'].replace({
        'exp0_builtin': '原始 BASE', 'stage2_builtin': '原始 BASE',
        'exp0_neutral': '中性 BASE', 'stage2_neutral': '中性 BASE',
        'stage2_selected': '选中参数',
    })
    return out.rename(columns=rename)


def compact_business_table(frame, split='eval'):
    cols = ['feature_group', 'model', 'experiment', 'beta', 'eta', 'kappa',
            f'AmountRecall_{split}_atK', f'MissedAmount_{split}_atK',
            f'MicroRecall_{split}_atK']
    rename = {'feature_group': '特征组', 'model': '模型', 'experiment': '实验',
              'beta': 'β', 'eta': 'η', 'kappa': 'κ',
              f'AmountRecall_{split}_atK': '金额召回@K',
              f'MissedAmount_{split}_atK': '漏报金额@K',
              f'MicroRecall_{split}_atK': '小额召回@K'}
    out = frame[[c for c in cols if c in frame.columns]].copy()
    out['experiment'] = out['experiment'].replace({
        'exp0_builtin': '原始 BASE', 'stage2_builtin': '原始 BASE',
        'exp0_neutral': '中性 BASE', 'stage2_neutral': '中性 BASE',
        'stage2_selected': '选中参数',
    })
    return out.rename(columns=rename)


def with_base_rows(frame, baseline_source):
    '''为每个结果表前置原始 BASE 与中性 BASE；只改变展示，不改变原始 CSV。'''
    if frame.empty:
        return frame.copy()
    base_names = ['exp0_builtin', 'exp0_neutral', 'stage2_builtin', 'stage2_neutral']
    wanted_models = set(frame['model'].astype(str))
    wanted_groups = set(frame['feature_group'].astype(str))
    base = baseline_source[
        baseline_source['experiment'].isin(base_names)
        & baseline_source['model'].astype(str).isin(wanted_models)
        & baseline_source['feature_group'].astype(str).isin(wanted_groups)
    ].copy()
    combined = pd.concat([base, frame], ignore_index=True, sort=False)
    key = [c for c in ['feature_group', 'model', 'experiment', 'loss_kind', 'beta', 'eta', 'kappa']
           if c in combined.columns]
    combined = combined.drop_duplicates(key, keep='first')
    combined['_display_order'] = combined['experiment'].map({
        'exp0_builtin': 0, 'stage2_builtin': 0,
        'exp0_neutral': 1, 'stage2_neutral': 1,
    }).fillna(2)
    combined = combined.sort_values(['feature_group', 'model', '_display_order', 'experiment'])
    return combined.drop(columns='_display_order').reset_index(drop=True)


## 5. 第一阶段粗网格：实验 0、1、2、3

- 实验 0：比较内置 balanced 与 neutral sample-weight。
- 实验 1：只搜索 beta。
- 实验 2：固定暂定 beta，搜索 eta。
- 实验 3：固定暂定 beta/eta，搜索 kappa。

实验 1–3 都在 neutral 路径上运行，所有 provisional 参数均按 calibration Top-K 指标选择。


In [140]:
# §5 prep：V2 OOF 缓存与运行单元
def cache_key(model_name, feature_group, loss_kind, beta, eta, kappa):
    raw = f'v2|{model_name}|{feature_group}|{loss_kind}|{beta:.8g}|{eta:.8g}|{kappa:.8g}'
    return hashlib.sha1(raw.encode('utf-8')).hexdigest()[:16]


def get_oof(model_name, data, feature_cols, feature_group, loss_kind,
            beta=0.0, eta=0.0, kappa=1.0):
    path = OOF_CACHE_DIR / f'{model_name}_{feature_group}_{loss_kind}_{cache_key(model_name, feature_group, loss_kind, beta, eta, kappa)}.npz'
    if path.is_file() and not FORCE_RERUN:
        saved = np.load(path, allow_pickle=False)
        return saved['oof'], saved['fold_ap'].tolist()
    if loss_kind == 'builtin':
        oof, fold_ap = cross_val_builtin_oof(model_name, data, feature_cols)
    else:
        oof, fold_ap, _ = cross_val_weighted_oof(model_name, data, feature_cols, beta, eta, kappa)
    np.savez_compressed(path, oof=oof, fold_ap=np.asarray(fold_ap, dtype=float))
    return oof, fold_ap


def run_config(model_name, data, feature_cols, feature_group, experiment, loss_kind,
               beta=0.0, eta=0.0, kappa=1.0, alert_rate=None):
    print(f'\n[{feature_group} | {model_name}] {experiment}: {loss_kind}, '
          f'beta={beta:.4g}, eta={eta:.4g}, kappa={kappa:.4g}')
    oof, fold_ap = get_oof(model_name, data, feature_cols, feature_group, loss_kind,
                           beta, eta, kappa)
    row = evaluate_oof_business(data, oof, fold_ap, alert_rate=alert_rate)
    row.update({'feature_group': feature_group, 'model': model_name,
                'experiment': experiment, 'loss_kind': loss_kind,
                'beta': float(beta), 'eta': float(eta), 'kappa': float(kappa)})
    return row


def best_on_cal(frame, model_name, experiments=None):
    part = frame[frame['model'] == model_name].copy()
    if experiments is not None:
        part = part[part['experiment'].isin(experiments)]
    return part.sort_values(
        ['AmountRecall_cal_atK', 'MicroRecall_cal_atK', 'AUC-PR_cal'],
        ascending=[False, False, False],
    ).iloc[0]


In [141]:
# §5 prep：粗网格实验 0–3
def run_stage1(data):
    rows = []
    for model_name in MODELS:
        anchor = run_config(model_name, data, BASE_FEATURES, 'BASE', 'exp0_builtin', 'builtin')
        rows.append(anchor)
        rate = anchor['alert_rate']
        neutral = run_config(model_name, data, BASE_FEATURES, 'BASE', 'exp0_neutral',
                             'amount_aware', 0, 0, 1, rate)
        rows.append(neutral)
        if abs(neutral['AUC-PR_mean'] - anchor['AUC-PR_mean']) > PARITY_AUC_TOL:
            print(f'警告：{model_name} neutral 与 builtin AUC-PR 差值超过 {PARITY_AUC_TOL}')

        for beta in BETA_GRID:
            rows.append(run_config(model_name, data, BASE_FEATURES, 'BASE', 'exp1_beta',
                                   'amount_aware', beta, 0, 1, rate))
        beta_best = float(best_on_cal(pd.DataFrame(rows), model_name, ['exp1_beta'])['beta'])
        for eta in ETA_GRID:
            rows.append(run_config(model_name, data, BASE_FEATURES, 'BASE', 'exp2_eta',
                                   'amount_aware', beta_best, eta, 1, rate))
        best2 = best_on_cal(pd.DataFrame(rows), model_name, ['exp1_beta', 'exp2_eta'])
        beta_best, eta_best = float(best2['beta']), float(best2['eta'])
        for kappa in KAPPA_GRID:
            rows.append(run_config(model_name, data, BASE_FEATURES, 'BASE', 'exp3_kappa',
                                   'amount_aware', beta_best, eta_best, kappa, rate))
        pd.DataFrame(rows).to_csv(OUT_COARSE / 'coarse_checkpoint.csv', index=False)
    result = pd.DataFrame(rows)
    result.to_csv(OUT_COARSE / 'coarse_results.csv', index=False)
    return result


In [142]:
# §5 exec：粗网格
if RUN_COARSE_GRID:
    coarse_results = run_stage1(df_all)
else:
    path = OUT_COARSE / 'coarse_results.csv'
    coarse_results = pd.read_csv(path) if path.is_file() else pd.DataFrame()
    print('RUN_COARSE_GRID=False；已有粗网格行数:', len(coarse_results))

if not coarse_results.empty:
    coarse_display = with_base_rows(coarse_results, coarse_results)
    print('分类核心表（calibration，用于选参）')
    display(compact_classification_table(coarse_display, 'cal').round(5))
    print('业务核心表（calibration，用于选参）')
    display(compact_business_table(coarse_display, 'cal').round(5))



[BASE | LightGBM] exp0_builtin: builtin, beta=0, eta=0, kappa=1
  LightGBM builtin fold 1/5 完成
  LightGBM builtin fold 2/5 完成
  LightGBM builtin fold 3/5 完成
  LightGBM builtin fold 4/5 完成
  LightGBM builtin fold 5/5 完成

[BASE | LightGBM] exp0_neutral: amount_aware, beta=0, eta=0, kappa=1
  LightGBM weighted fold 1/5 完成 (beta=0, eta=0, kappa=1)
  LightGBM weighted fold 2/5 完成 (beta=0, eta=0, kappa=1)
  LightGBM weighted fold 3/5 完成 (beta=0, eta=0, kappa=1)
  LightGBM weighted fold 4/5 完成 (beta=0, eta=0, kappa=1)
  LightGBM weighted fold 5/5 完成 (beta=0, eta=0, kappa=1)

[BASE | LightGBM] exp1_beta: amount_aware, beta=0.25, eta=0, kappa=1
  LightGBM weighted fold 1/5 完成 (beta=0.25, eta=0, kappa=1)
  LightGBM weighted fold 2/5 完成 (beta=0.25, eta=0, kappa=1)
  LightGBM weighted fold 3/5 完成 (beta=0.25, eta=0, kappa=1)
  LightGBM weighted fold 4/5 完成 (beta=0.25, eta=0, kappa=1)
  LightGBM weighted fold 5/5 完成 (beta=0.25, eta=0, kappa=1)

[BASE | LightGBM] exp1_beta: amount_aware, beta=0.5, e

,特征组,模型,实验,损失,β,η,κ,AUC-PR,F1@K,Precision@K,Recall@K,FP@K,FN@K
0,BASE,LightGBM,原始 BASE,builtin,0.00,0.00,1.00,0.77364,0.82353,0.94030,0.73256,4,23
1,BASE,LightGBM,中性 BASE,amount_aware,0.00,0.00,1.00,0.76649,0.81046,0.92537,0.72093,5,24
2,BASE,LightGBM,exp1_beta,amount_aware,0.25,0.00,1.00,0.77099,0.81046,0.92537,0.72093,5,24
3,BASE,LightGBM,exp1_beta,amount_aware,0.50,0.00,1.00,0.76860,0.81046,0.92537,0.72093,5,24
4,BASE,LightGBM,exp1_beta,amount_aware,1.00,0.00,1.00,0.76367,0.81046,0.92537,0.72093,5,24
5,BASE,LightGBM,exp1_beta,amount_aware,2.00,0.00,1.00,0.76489,0.79739,0.91045,0.70930,6,25
6,BASE,LightGBM,exp1_beta,amount_aware,4.00,0.00,1.00,0.76227,0.79739,0.91045,0.70930,6,25
7,BASE,LightGBM,exp2_eta,amount_aware,0.25,0.25,1.00,0.76937,0.79739,0.91045,0.70930,6,25
8,BASE,LightGBM,exp2_eta,amount_aware,0.25,0.50,1.00,0.77074,0.81046,0.92537,0.72093,5,24
9,BASE,LightGBM,exp2_eta,amount_aware,0.25,1.00,1.00,0.77276,0.82353,0.94030,0.73256,4,23


业务核心表（calibration，用于选参）


,特征组,模型,实验,β,η,κ,金额召回@K,漏报金额@K,小额召回@K
0,BASE,LightGBM,原始 BASE,0.00,0.00,1.00,0.76980,2782.79,0.81250
1,BASE,LightGBM,中性 BASE,0.00,0.00,1.00,0.76971,2783.79,0.78125
2,BASE,LightGBM,exp1_beta,0.25,0.00,1.00,0.76260,2869.79,0.81250
3,BASE,LightGBM,exp1_beta,0.50,0.00,1.00,0.76260,2869.79,0.81250
4,BASE,LightGBM,exp1_beta,1.00,0.00,1.00,0.73619,3188.99,0.84375
5,BASE,LightGBM,exp1_beta,2.00,0.00,1.00,0.73571,3194.90,0.84375
6,BASE,LightGBM,exp1_beta,4.00,0.00,1.00,0.73611,3189.99,0.81250
7,BASE,LightGBM,exp2_eta,0.25,0.25,1.00,0.76203,2876.70,0.81250
8,BASE,LightGBM,exp2_eta,0.25,0.50,1.00,0.76988,2781.79,0.78125
9,BASE,LightGBM,exp2_eta,0.25,1.00,1.00,0.76988,2781.79,0.81250


## 6. 局部精细网格

在粗网格最优点附近依次细化 beta、eta、kappa，每一步只改变一个参数。这样比 5×5×5 全笛卡尔积更节省计算，也更容易解释是哪一个参数带来变化。


In [143]:
# §6 prep：局部精细网格
def local_values(center, name):
    center = float(center)
    if name in ('beta', 'eta'):
        if center <= 1e-12:
            return [0.0, 0.05, 0.1, 0.2, 0.35]
        step = max(0.05, center * 0.25)
        return sorted(set(round(max(0.0, center + offset * step), 6) for offset in [-2, -1, 0, 1, 2]))
    step = max(0.05, center * 0.1)
    return sorted(set(round(float(np.clip(center + offset * step, 0.25, 2.0)), 6)
                      for offset in [-2, -1, 0, 1, 2]))


def run_fine_grid(data, coarse):
    rows = []
    for model_name in MODELS:
        anchor = coarse[(coarse.model == model_name) & (coarse.experiment == 'exp0_builtin')].iloc[0]
        rate = float(anchor['alert_rate'])
        base = best_on_cal(coarse[coarse.loss_kind == 'amount_aware'], model_name,
                           ['exp1_beta', 'exp2_eta', 'exp3_kappa'])
        beta, eta, kappa = float(base.beta), float(base.eta), float(base.kappa)
        for value in local_values(beta, 'beta'):
            rows.append(run_config(model_name, data, BASE_FEATURES, 'BASE', 'fine_beta',
                                   'amount_aware', value, eta, kappa, rate))
        beta = float(best_on_cal(pd.DataFrame(rows), model_name, ['fine_beta']).beta)
        for value in local_values(eta, 'eta'):
            rows.append(run_config(model_name, data, BASE_FEATURES, 'BASE', 'fine_eta',
                                   'amount_aware', beta, value, kappa, rate))
        best_eta = best_on_cal(pd.DataFrame(rows), model_name, ['fine_beta', 'fine_eta'])
        beta, eta = float(best_eta.beta), float(best_eta.eta)
        for value in local_values(kappa, 'kappa'):
            rows.append(run_config(model_name, data, BASE_FEATURES, 'BASE', 'fine_kappa',
                                   'amount_aware', beta, eta, value, rate))
        pd.DataFrame(rows).to_csv(OUT_FINE / 'fine_grid_checkpoint.csv', index=False)
    result = pd.DataFrame(rows)
    result.to_csv(OUT_FINE / 'fine_grid_results.csv', index=False)
    return result


In [144]:
# §6 exec：局部精细网格
if RUN_FINE_GRID:
    if coarse_results.empty:
        raise RuntimeError('请先完成粗网格')
    fine_results = run_fine_grid(df_all, coarse_results)
else:
    path = OUT_FINE / 'fine_grid_results.csv'
    fine_results = pd.read_csv(path) if path.is_file() else pd.DataFrame()
    print('RUN_FINE_GRID=False；已有精细网格行数:', len(fine_results))
if not fine_results.empty:
    fine_display = with_base_rows(fine_results, coarse_results)
    display(compact_business_table(fine_display, 'cal').round(5))



[BASE | LightGBM] fine_beta: amount_aware, beta=0.125, eta=2, kappa=1
  LightGBM weighted fold 1/5 完成 (beta=0.125, eta=2, kappa=1)
  LightGBM weighted fold 2/5 完成 (beta=0.125, eta=2, kappa=1)
  LightGBM weighted fold 3/5 完成 (beta=0.125, eta=2, kappa=1)
  LightGBM weighted fold 4/5 完成 (beta=0.125, eta=2, kappa=1)
  LightGBM weighted fold 5/5 完成 (beta=0.125, eta=2, kappa=1)

[BASE | LightGBM] fine_beta: amount_aware, beta=0.1875, eta=2, kappa=1
  LightGBM weighted fold 1/5 完成 (beta=0.1875, eta=2, kappa=1)
  LightGBM weighted fold 2/5 完成 (beta=0.1875, eta=2, kappa=1)
  LightGBM weighted fold 3/5 完成 (beta=0.1875, eta=2, kappa=1)
  LightGBM weighted fold 4/5 完成 (beta=0.1875, eta=2, kappa=1)
  LightGBM weighted fold 5/5 完成 (beta=0.1875, eta=2, kappa=1)

[BASE | LightGBM] fine_beta: amount_aware, beta=0.25, eta=2, kappa=1
  LightGBM weighted fold 1/5 完成 (beta=0.25, eta=2, kappa=1)
  LightGBM weighted fold 2/5 完成 (beta=0.25, eta=2, kappa=1)
  LightGBM weighted fold 3/5 完成 (beta=0.25, eta=2, k

,特征组,模型,实验,β,η,κ,金额召回@K,漏报金额@K,小额召回@K
0,BASE,LightGBM,原始 BASE,0.0000,0.00,1.0,0.76980,2782.79,0.81250
1,BASE,LightGBM,中性 BASE,0.0000,0.00,1.0,0.76971,2783.79,0.78125
2,BASE,LightGBM,fine_beta,0.1250,2.00,1.0,0.76988,2781.79,0.78125
3,BASE,LightGBM,fine_beta,0.1875,2.00,1.0,0.76988,2781.79,0.78125
4,BASE,LightGBM,fine_beta,0.2500,2.00,1.0,0.77151,2762.06,0.78125
5,BASE,LightGBM,fine_beta,0.3125,2.00,1.0,0.76988,2781.79,0.81250
6,BASE,LightGBM,fine_beta,0.3750,2.00,1.0,0.76988,2781.79,0.81250
7,BASE,LightGBM,fine_eta,0.2500,1.00,1.0,0.76988,2781.79,0.81250
8,BASE,LightGBM,fine_eta,0.2500,1.50,1.0,0.76939,2787.70,0.81250
9,BASE,LightGBM,fine_eta,0.2500,2.00,1.0,0.77151,2762.06,0.78125


## 7. Optuna 多目标搜索

使用 NSGA-II 同时最大化 calibration 段的：

1. `AmountRecall@K`；
2. `MicroRecall@K`；
3. AUC-PR。

搜索空间为 beta∈[0,6]、eta∈[0,6]、kappa∈[0.4,1.6]。默认每模型 30 trials，共约 300 次五折模型拟合；结果和 OOF 均可断点复用。多目标搜索不会保证存在优于 baseline 的参数，它只会更系统地描出可达到的 Pareto 边界。


In [145]:
# §7 prep：Optuna NSGA-II 多目标搜索
def run_optuna_multiobjective(data, coarse, n_trials=OPTUNA_N_TRIALS):
    all_rows, selected = [], {}
    for model_name in MODELS:
        anchor = coarse[(coarse.model == model_name) & (coarse.experiment == 'exp0_builtin')].iloc[0]
        rate = float(anchor['alert_rate'])

        def objective(trial):
            beta = trial.suggest_float('beta', 0.0, 6.0)
            eta = trial.suggest_float('eta', 0.0, 6.0)
            kappa = trial.suggest_float('kappa', 0.4, 1.6)
            row = run_config(model_name, data, BASE_FEATURES, 'BASE', 'optuna',
                             'amount_aware', beta, eta, kappa, rate)
            row['trial_number'] = trial.number
            all_rows.append(row)
            for key, value in row.items():
                if isinstance(value, (str, int, float, np.integer, np.floating)):
                    trial.set_user_attr(key, value.item() if hasattr(value, 'item') else value)
            return row['AmountRecall_cal_atK'], row['MicroRecall_cal_atK'], row['AUC-PR_cal']

        study = optuna.create_study(
            directions=['maximize', 'maximize', 'maximize'],
            sampler=optuna.samplers.NSGAIISampler(seed=CV_RANDOM_STATE),
            study_name=f'amount_loss_v2_{model_name}',
        )
        study.optimize(objective, n_trials=int(n_trials), show_progress_bar=True)
        trials = pd.DataFrame([t.user_attrs for t in study.trials if t.user_attrs])
        pareto_numbers = {t.number for t in study.best_trials}
        trials['pareto'] = trials['trial_number'].isin(pareto_numbers)
        trials.to_csv(OUT_OPTUNA / f'optuna_trials_{model_name}.csv', index=False)
        best = best_on_cal(trials[trials.pareto], model_name)
        selected[model_name] = {'beta': float(best.beta), 'eta': float(best.eta),
                                'kappa': float(best.kappa), 'source': 'optuna_pareto'}
    result = pd.DataFrame(all_rows)
    result.to_csv(OUT_OPTUNA / 'optuna_results.csv', index=False)
    (OUT_OPTUNA / 'optuna_selected.json').write_text(
        json.dumps(selected, ensure_ascii=False, indent=2), encoding='utf-8')
    return result, selected


In [146]:
# §7 exec：Optuna
if RUN_OPTUNA:
    if coarse_results.empty:
        raise RuntimeError('请先完成粗网格')
    optuna_results, optuna_selected = run_optuna_multiobjective(df_all, coarse_results)
else:
    path = OUT_OPTUNA / 'optuna_results.csv'
    optuna_results = pd.read_csv(path) if path.is_file() else pd.DataFrame()
    selected_path = OUT_OPTUNA / 'optuna_selected.json'
    optuna_selected = json.loads(selected_path.read_text(encoding='utf-8')) if selected_path.is_file() else {}
    print('RUN_OPTUNA=False；已有 Optuna 行数:', len(optuna_results))
if not optuna_results.empty:
    optuna_display = with_base_rows(optuna_results, coarse_results)
    display(compact_business_table(optuna_display, 'cal').round(5))


[I 2026-07-07 22:02:14,029] A new study created in memory with name: amount_loss_v2_LightGBM


  0%|          | 0/30 [00:00<?, ?it/s]


[BASE | LightGBM] optuna: amount_aware, beta=2.247, eta=5.704, kappa=1.278
  LightGBM weighted fold 1/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
  LightGBM weighted fold 2/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
  LightGBM weighted fold 3/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
  LightGBM weighted fold 4/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
  LightGBM weighted fold 5/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
[I 2026-07-07 22:02:22,095] Trial 0 finished with values: [0.7693075752850463, 0.8125, 0.7694979189405683] and parameters: {'beta': 2.247240713084175, 'eta': 5.704285838459497, 'kappa': 1.2783927301736862}.

[BASE | LightGBM] optuna: amount_aware, beta=3.592, eta=0.9361, kappa=0.5872
  LightGBM weighted fold 1/5 完成 (beta=3.592, eta=0.9361, kappa=0.5872)
  LightGBM weighted fold 2/5 完成 (beta=3.592, eta=0.9361, kappa=0.5872)
  LightGBM weighted fold 3/5 完成 (beta=3.592, eta=0.9361, kappa=0.5872)
  LightGBM weighted fold 4/5 完成 (beta=3.592, eta=0.9361, kappa=0.5872)
  LightGBM

[I 2026-07-07 22:06:10,462] A new study created in memory with name: amount_loss_v2_XGBoost


  LightGBM weighted fold 5/5 完成 (beta=3.825, eta=5.323, kappa=0.9667)
[I 2026-07-07 22:06:10,454] Trial 29 finished with values: [0.7698791981397025, 0.8125, 0.7710285332389715] and parameters: {'beta': 3.8253448281312785, 'eta': 5.323276455457959, 'kappa': 0.9666579101943392}.


  0%|          | 0/30 [00:00<?, ?it/s]


[BASE | XGBoost] optuna: amount_aware, beta=2.247, eta=5.704, kappa=1.278
  XGBoost weighted fold 1/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
  XGBoost weighted fold 2/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
  XGBoost weighted fold 3/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
  XGBoost weighted fold 4/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
  XGBoost weighted fold 5/5 完成 (beta=2.247, eta=5.704, kappa=1.278)
[I 2026-07-07 22:06:15,638] Trial 0 finished with values: [0.7610244209526663, 0.8125, 0.7532252508345361] and parameters: {'beta': 2.247240713084175, 'eta': 5.704285838459497, 'kappa': 1.2783927301736862}.

[BASE | XGBoost] optuna: amount_aware, beta=3.592, eta=0.9361, kappa=0.5872
  XGBoost weighted fold 1/5 完成 (beta=3.592, eta=0.9361, kappa=0.5872)
  XGBoost weighted fold 2/5 完成 (beta=3.592, eta=0.9361, kappa=0.5872)
  XGBoost weighted fold 3/5 完成 (beta=3.592, eta=0.9361, kappa=0.5872)
  XGBoost weighted fold 4/5 完成 (beta=3.592, eta=0.9361, kappa=0.5872)
  XGBoost weighted fo

,特征组,模型,实验,β,η,κ,金额召回@K,漏报金额@K,小额召回@K
0,BASE,LightGBM,原始 BASE,0.00000,0.00000,1.00000,0.76980,2782.79,0.81250
1,BASE,LightGBM,中性 BASE,0.00000,0.00000,1.00000,0.76971,2783.79,0.78125
2,BASE,LightGBM,optuna,2.24724,5.70429,1.27839,0.76931,2788.70,0.81250
3,BASE,LightGBM,optuna,3.59195,0.93611,0.58719,0.76252,2870.79,0.78125
4,BASE,LightGBM,optuna,0.34850,5.19706,1.12134,0.79154,2519.92,0.78125
...,...,...,...,...,...,...,...,...,...
59,BASE,XGBoost,optuna,4.37404,4.62762,0.48885,0.76260,2869.79,0.81250
60,BASE,XGBoost,optuna,2.15079,0.69521,1.43572,0.73619,3188.99,0.81250
61,BASE,XGBoost,optuna,3.73979,1.98539,0.47627,0.76260,2869.79,0.81250
62,BASE,XGBoost,optuna,1.86589,1.95110,1.27553,0.73619,3188.99,0.84375


## 8. 合并候选并只用 calibration 选最终参数

粗网格、精细网格和 Optuna 候选统一放入 calibration Pareto 比较。最终参数保存后，才读取其 evaluation 指标并进入第二阶段。


In [147]:
# §8 prep：Pareto 与最终选参
def pareto_front(frame):
    frame = frame.reset_index(drop=True).copy()
    values = frame[['MissedAmount_cal_atK', 'MicroFN_cal_atK', 'FN_cal_atK']].to_numpy(float)
    keep = np.ones(len(frame), dtype=bool)
    for i in range(len(frame)):
        dominated = np.all(values <= values[i], axis=1) & np.any(values < values[i], axis=1)
        dominated[i] = False
        keep[i] = not dominated.any()
    return frame[keep].copy()


def select_final_params(coarse, fine, optuna_frame):
    frames = [x for x in (coarse, fine, optuna_frame) if not x.empty]
    candidates = pd.concat(frames, ignore_index=True)
    candidates = candidates[candidates.loss_kind == 'amount_aware'].copy()
    selected, fronts = {}, []
    for model_name in MODELS:
        front = pareto_front(candidates[candidates.model == model_name])
        front['pareto'] = True
        fronts.append(front)
        best = best_on_cal(front, model_name)
        selected[model_name] = {'beta': float(best.beta), 'eta': float(best.eta),
                                'kappa': float(best.kappa), 'source': str(best.experiment)}
    return selected, pd.concat(fronts, ignore_index=True)


In [148]:
# §8 exec：最终 calibration 选参
if not coarse_results.empty:
    FINAL_SELECTED_PARAMS, final_pareto = select_final_params(
        coarse_results, fine_results, optuna_results)
    (OUT_AMOUNT_LOSS / 'final_selected_params.json').write_text(
        json.dumps(FINAL_SELECTED_PARAMS, ensure_ascii=False, indent=2), encoding='utf-8')
    final_pareto.to_csv(OUT_AMOUNT_LOSS / 'final_calibration_pareto.csv', index=False)
    print(json.dumps(FINAL_SELECTED_PARAMS, ensure_ascii=False, indent=2))
    pareto_display = with_base_rows(final_pareto, coarse_results)
    display(compact_business_table(pareto_display, 'cal').round(5))
else:
    FINAL_SELECTED_PARAMS, final_pareto = {}, pd.DataFrame()


{
  "LightGBM": {
    "beta": 0.34850167300919677,
    "eta": 5.197056874649611,
    "kappa": 1.1213380140918507,
    "source": "optuna"
  },
  "XGBoost": {
    "beta": 4.248435466776273,
    "eta": 0.12350696577481468,
    "kappa": 1.5638918225943934,
    "source": "optuna"
  }
}


,特征组,模型,实验,β,η,κ,金额召回@K,漏报金额@K,小额召回@K
0,BASE,LightGBM,原始 BASE,0.00000,0.00000,1.00000,0.76980,2782.79,0.81250
1,BASE,LightGBM,中性 BASE,0.00000,0.00000,1.00000,0.76971,2783.79,0.78125
2,BASE,LightGBM,exp1_beta,1.00000,0.00000,1.00000,0.73619,3188.99,0.84375
3,BASE,LightGBM,exp2_eta,0.25000,1.00000,1.00000,0.76988,2781.79,0.81250
4,BASE,LightGBM,exp3_kappa,0.25000,2.00000,1.25000,0.76988,2781.79,0.81250
5,BASE,LightGBM,fine_beta,0.31250,2.00000,1.00000,0.76988,2781.79,0.81250
6,BASE,LightGBM,fine_beta,0.37500,2.00000,1.00000,0.76988,2781.79,0.81250
7,BASE,LightGBM,fine_eta,0.25000,1.00000,1.00000,0.76988,2781.79,0.81250
8,BASE,LightGBM,fine_eta,0.25000,2.50000,1.00000,0.76988,2781.79,0.81250
9,BASE,LightGBM,optuna,0.34850,5.19706,1.12134,0.79154,2519.92,0.78125


## 9. 第二阶段：BASE / §7–8 胜者 × neutral / 新损失

每个特征组同时保留 builtin 锚点、neutral 因果 baseline 和选中的新损失。参数由 BASE calibration 选出；第二阶段 evaluation 不再参与选参。


In [149]:
# §9 prep：第二阶段迁移
def load_final_params():
    path = OUT_AMOUNT_LOSS / 'final_selected_params.json'
    if not path.is_file():
        raise FileNotFoundError('请先完成 §8 最终选参')
    return json.loads(path.read_text(encoding='utf-8'))


def run_stage2(data):
    selected = load_final_params()
    groups = {'BASE': BASE_FEATURES, 'ABLATION_WINNER': WINNER_FEATURES}
    rows = []
    for group, features in groups.items():
        for model_name in MODELS:
            anchor = run_config(model_name, data, features, group, 'stage2_builtin', 'builtin')
            rows.append(anchor)
            rate = anchor['alert_rate']
            rows.append(run_config(model_name, data, features, group, 'stage2_neutral',
                                   'amount_aware', 0, 0, 1, rate))
            p = selected[model_name]
            rows.append(run_config(model_name, data, features, group, 'stage2_selected',
                                   'amount_aware', p['beta'], p['eta'], p['kappa'], rate))
            pd.DataFrame(rows).to_csv(OUT_STAGE2 / 'stage2_checkpoint.csv', index=False)
    result = pd.DataFrame(rows)
    result.to_csv(OUT_STAGE2 / 'stage2_results.csv', index=False)
    return result


In [150]:
# §9 exec：第二阶段
if RUN_STAGE2:
    stage2_results = run_stage2(df_all)
else:
    path = OUT_STAGE2 / 'stage2_results.csv'
    stage2_results = pd.read_csv(path) if path.is_file() else pd.DataFrame()
    print('RUN_STAGE2=False；已有第二阶段行数:', len(stage2_results))
if not stage2_results.empty:
    stage2_display = with_base_rows(stage2_results, stage2_results)
    print('分类核心表（evaluation）')
    display(compact_classification_table(stage2_display, 'eval').round(5))
    print('业务核心表（evaluation）')
    display(compact_business_table(stage2_display, 'eval').round(5))



[BASE | LightGBM] stage2_builtin: builtin, beta=0, eta=0, kappa=1
  LightGBM builtin fold 1/5 完成
  LightGBM builtin fold 2/5 完成
  LightGBM builtin fold 3/5 完成
  LightGBM builtin fold 4/5 完成
  LightGBM builtin fold 5/5 完成

[BASE | LightGBM] stage2_neutral: amount_aware, beta=0, eta=0, kappa=1
  LightGBM weighted fold 1/5 完成 (beta=0, eta=0, kappa=1)
  LightGBM weighted fold 2/5 完成 (beta=0, eta=0, kappa=1)
  LightGBM weighted fold 3/5 完成 (beta=0, eta=0, kappa=1)
  LightGBM weighted fold 4/5 完成 (beta=0, eta=0, kappa=1)
  LightGBM weighted fold 5/5 完成 (beta=0, eta=0, kappa=1)

[BASE | LightGBM] stage2_selected: amount_aware, beta=0.3485, eta=5.197, kappa=1.121
  LightGBM weighted fold 1/5 完成 (beta=0.3485, eta=5.197, kappa=1.121)
  LightGBM weighted fold 2/5 完成 (beta=0.3485, eta=5.197, kappa=1.121)
  LightGBM weighted fold 3/5 完成 (beta=0.3485, eta=5.197, kappa=1.121)
  LightGBM weighted fold 4/5 完成 (beta=0.3485, eta=5.197, kappa=1.121)
  LightGBM weighted fold 5/5 完成 (beta=0.3485, eta=5.197

,特征组,模型,实验,损失,β,η,κ,AUC-PR,F1@K,Precision@K,Recall@K,FP@K,FN@K
0,ABLATION_WINNER,LightGBM,原始 BASE,builtin,0.00000,0.00000,1.00000,0.77073,0.77447,0.86667,0.70000,28,78
1,ABLATION_WINNER,LightGBM,中性 BASE,amount_aware,0.00000,0.00000,1.00000,0.77129,0.78298,0.87619,0.70769,26,76
2,ABLATION_WINNER,LightGBM,选中参数,amount_aware,0.34850,5.19706,1.12134,0.76873,0.76170,0.85238,0.68846,31,81
3,ABLATION_WINNER,XGBoost,原始 BASE,builtin,0.00000,0.00000,1.00000,0.74994,0.74790,0.82407,0.68462,38,82
4,ABLATION_WINNER,XGBoost,中性 BASE,amount_aware,0.00000,0.00000,1.00000,0.74605,0.73950,0.81481,0.67692,40,84
5,ABLATION_WINNER,XGBoost,选中参数,amount_aware,4.24844,0.12351,1.56389,0.72951,0.70168,0.77315,0.64231,49,93
6,BASE,LightGBM,原始 BASE,builtin,0.00000,0.00000,1.00000,0.75902,0.75922,0.87065,0.67308,26,85
7,BASE,LightGBM,中性 BASE,amount_aware,0.00000,0.00000,1.00000,0.76323,0.77657,0.89055,0.68846,22,81
8,BASE,LightGBM,选中参数,amount_aware,0.34850,5.19706,1.12134,0.77149,0.76356,0.87562,0.67692,25,84
9,BASE,XGBoost,原始 BASE,builtin,0.00000,0.00000,1.00000,0.74061,0.74043,0.82857,0.66923,36,86


业务核心表（evaluation）


,特征组,模型,实验,β,η,κ,金额召回@K,漏报金额@K,小额召回@K
0,ABLATION_WINNER,LightGBM,原始 BASE,0.00000,0.00000,1.00000,0.60995,12999.18,0.68182
1,ABLATION_WINNER,LightGBM,中性 BASE,0.00000,0.00000,1.00000,0.61319,12891.44,0.68182
2,ABLATION_WINNER,LightGBM,选中参数,0.34850,5.19706,1.12134,0.61172,12940.13,0.70455
3,ABLATION_WINNER,XGBoost,原始 BASE,0.00000,0.00000,1.00000,0.60563,13143.40,0.68182
4,ABLATION_WINNER,XGBoost,中性 BASE,0.00000,0.00000,1.00000,0.58913,13693.08,0.67045
5,ABLATION_WINNER,XGBoost,选中参数,4.24844,0.12351,1.56389,0.57605,14128.91,0.63636
6,BASE,LightGBM,原始 BASE,0.00000,0.00000,1.00000,0.61870,12707.66,0.63636
7,BASE,LightGBM,中性 BASE,0.00000,0.00000,1.00000,0.61174,12939.46,0.65909
8,BASE,LightGBM,选中参数,0.34850,5.19706,1.12134,0.60312,13226.91,0.65909
9,BASE,XGBoost,原始 BASE,0.00000,0.00000,1.00000,0.58063,13976.56,0.69318


## 10. 结果解释

- `金额召回@K` 越高越好；`漏报金额@K` 越低越好，两者表达同一业务结果，因此窄表可以按阅读偏好保留其中一个，完整 CSV 两者都保存。
- `小额召回@K` 用于检查卡测试保护，不再预设任意下降百分点。
- Top-K 是离线排序能力评价；生产部署仍需要在独立 calibration 数据上把 K 对应回阈值并监控漂移。
- 增加搜索精细度只能更充分探索当前损失函数族，不能保证一定超过 baseline。若最终 evaluation 仍没有 Pareto 改善，应接受“该损失设计在此数据上无增益”的结论，而不是继续扩大搜索直到撞上偶然高分。
